Install all dependencies

In [ ]:
!pip install torch transformers ranx accelerate

Download the collections

In [ ]:
!wget "https://uapt33090-my.sharepoint.com/:u:/g/personal/joao_fonseca_ua_pt/EaBHP-PszZhBr3f4dfAFA3MBAE_XTB6k-iW4mgUf5dYjbg?e=GFuQLT&download=1" -O pubmed_2022_tiny.jsonl.gz
!wget "https://uapt33090-my.sharepoint.com/:u:/g/personal/joao_fonseca_ua_pt/EXZIa3anvQ5DmTZ9MspBspMB6IIRORb1Wrb_a9lTb3GbIA?e=Ngdgai&download=1" -O pubmed_2022_small.jsonl.gz

Import all modules

In [ ]:
import gzip
import json
import math
import torch

from collator import RankingCollator
from collections import defaultdict
from data import BioASQDataset, BioASQPointwiseIterator, InferenceRankingIterator, InferenceDataset
from google.colab import drive
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments
from ranker_trainer import RankerTrainer
from sampler import BasicSampler
from sklearn.metrics import precision_score, recall_score
from tqdm import tqdm
from utils import load_collection_lookup

# Train the model

Clone GitHub repository and copy data and trainer files

In [ ]:
!git clone https://github.com/joaompfonseca/ri-neural-reranker.git
!cp ri-neural-reranker/data/* .
!cp ri-neural-reranker/trainer/* .

Preview the train dataset

In [ ]:
import json

with open("train_dataset.jsonl") as train_dataset:
  for line in train_dataset:
    data = json.loads(line)
    print(data["query"])
    print(data["pos_docs"])
    print(data["neg_docs"])
    break

Choose the model checkpoint

In [ ]:
model_checkpoint = "bert-base-uncased"

Choose the device

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Configure the model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2).to("cuda")

Configure the tokenizer

In [ ]:
TOKENIZER_LENGTH = 512

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.model_max_length = TOKENIZER_LENGTH

Prepare collection, dataset and relevant documents to feed into dataset

In [ ]:
MAX_QUERIES = 500 # out of 2471

In [ ]:
collection = dict()
dataset = dict()
qrels = dict()

# Collection - Tiny because train dataset negatives are taken from it
with gzip.open("pubmed_2022_tiny.jsonl.gz", "r") as all_file:
  for i, line in enumerate(all_file):
    print(f"Loading {i} documents from tiny dataset...", end="\r")
    doc = json.loads(line)
    collection[doc["pmid"]] = doc["title"] + " " + doc["abstract"]

# Dataset and relevant documents
with open("train_dataset.jsonl") as train_dataset:
  for line in train_dataset:
    data = json.loads(line)
    query = data["query"]
    for i in range(len(data["pos_docs"])):
      collection[data["pos_docs"][i]["id"]] = data["pos_docs"][i]["text"]
      collection[data["neg_docs"][i]["id"]] = data["neg_docs"][i]["text"]
    pos_docs = [doc["id"] for doc in data["pos_docs"]]
    neg_docs = [doc["id"] for doc in data["neg_docs"]]
    dataset[query] = {"question": query, "pos_docs": pos_docs, "neg_docs": neg_docs}
    qrels[query] = {docid: 1 for docid in pos_docs}

train_dataset = BioASQDataset(
  dataset=dataset,
  tokenizer=tokenizer,
  qrels_dict=qrels,
  collection=collection,
  iterator_class=BioASQPointwiseIterator[BasicSampler],
  max_questions=MAX_QUERIES
)

Configure the training arguments

In [ ]:
BATCH_SIZE       = 8
LEARNING_RATE    = 2e-5 # AdamW
NUMBER_OF_EPOCHS = 5

In [ ]:
training_args = TrainingArguments(
  num_train_epochs=NUMBER_OF_EPOCHS,
  learning_rate=LEARNING_RATE,
  weight_decay=0.01,
  per_device_train_batch_size=BATCH_SIZE,
  dataloader_pin_memory=True,
  output_dir="train_output",
  logging_strategy="steps",
  logging_first_step=True,
  logging_steps=100,
  save_strategy="epoch",
  save_total_limit=2,
  seed=42
)

trainer = RankerTrainer(
  model=model,
  args=training_args,
  train_dataset=train_dataset,
  tokenizer=tokenizer,
  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
  preprocess_logits_for_metrics=lambda logits, labels: torch.nn.functional.softmax(logits, dim=-1)[:,1],
)

Train the model!

In [ ]:
trainer.train()

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Token indices sequence length is longer than the specified maximum sequence length for this model (755 > 512). Running this sequence through the model will result in indexing errors


Step,Training Loss
1,0.703800
100,0.201300
200,0.027300
300,0.057700
400,0.009000
500,0.038700
600,0.014100
700,0.023400
800,0.017600
900,0.033000


TrainOutput(global_step=4760, training_loss=0.017651909840598452, metrics={'train_runtime': 3340.6882, 'train_samples_per_second': 11.396, 'train_steps_per_second': 1.425, 'total_flos': 8891281060453680.0, 'train_loss': 0.017651909840598452, 'epoch': 5.0})

Zip the train results and mount Drive to copy them there

In [ ]:
!zip -r train_output.zip train_output/
drive.mount('/content/drive')
!cp train_output.zip drive/train_output.zip

# Rerank BM25 using the model

Download the model

In [ ]:
!wget "https://uapt33090-my.sharepoint.com/:u:/g/personal/joao_fonseca_ua_pt/ERPQDz2GemFNr_2IUBa8mtYBN3URb6orjW73vHnLjxt87Q?e=HEdAeL&download=1" -O train_output.zip
!unzip train_output.zip

Choose the model checkpoint

In [2]:
model_checkpoint = "train_output/checkpoint-4760"

Choose the device

In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

Configure the model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint).to("cuda")

Configure the tokenizer

In [ ]:
TOKENIZER_LENGTH = 512

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.model_max_length = TOKENIZER_LENGTH

Load tiny collection for BM25_E9B1 runs

In [ ]:
BATCH_SIZE = 32

In [ ]:
collection = dict()
with gzip.open("pubmed_2022_tiny.jsonl.gz", "r") as all_file:
  for i, line in enumerate(all_file):
    print(f"Loading {i} documents from tiny dataset...", end="\r")
    doc = json.loads(line)
    collection[doc["pmid"]] = doc["title"] + " " + doc["abstract"]

bm25_dataset = InferenceDataset(
  "BM25_E9B1.jsonl",
  collection,
  tokenizer,
  at=100,
  iterator_class=InferenceRankingIterator
)

dataloader = torch.utils.data.DataLoader(
  bm25_dataset,
  batch_size=BATCH_SIZE,
  pin_memory=True,
  collate_fn=RankingCollator(tokenizer)
)

Rerank BM_25 results using the model!

In [ ]:
bm25_rerank = defaultdict(list)

for sample in tqdm(dataloader):
  _inputs = sample["inputs"].to("cuda")

  with torch.no_grad():
    logits = model(**_inputs).logits
    score = torch.nn.functional.softmax(logits, dim=-1)[:,1] # [0-1]

  for i, q_id in enumerate(sample["id"]):
    bm25_rerank[q_id].append({"id":sample["doc_id"][i],
                           "score":score[i].item()})

# Sort documents by relevance
for q_id in bm25_rerank:
  bm25_rerank[q_id].sort(key=lambda x:-x["score"])

In [ ]:
bm25_rerank

defaultdict(list,
            {'601bde6e1cb411341a000006': [{'id': '33186545',
               'score': 0.9999885559082031},
              {'id': '22495306', 'score': 0.9999761581420898},
              {'id': '34158173', 'score': 0.9999752044677734},
              {'id': '24504326', 'score': 0.99996018409729},
              {'id': '29790814', 'score': 0.9999597072601318},
              {'id': '20436468', 'score': 0.9999587535858154},
              {'id': '26749308', 'score': 0.9999508857727051},
              {'id': '25773295', 'score': 0.9999501705169678},
              {'id': '11535573', 'score': 0.9999494552612305},
              {'id': '24747641', 'score': 0.9999494552612305},
              {'id': '21907581', 'score': 0.9999483823776245},
              {'id': '29478617', 'score': 0.9999432563781738},
              {'id': '34301630', 'score': 0.9999412298202515},
              {'id': '8078586', 'score': 0.9999412298202515},
              {'id': '32910914', 'score': 0.9999395608901978